In [27]:
import sys
sys.path.insert(0, '/Users/vahid/Downloads/FERNN-master/moving_mnist_fp')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Import Moving MNIST dataset
from moving_mnist_dataset import MovingMNISTDataset

# Force reload of models module
import importlib
import moving_mnist_models
importlib.reload(moving_mnist_models)

print("✓ All imports successful")

✓ All imports successful


Output shape: torch.Size([2, 10, 1, 28, 28])
Velocity probs shape: torch.Size([2, 20, 25])
Velocity probs sum (first batch, first step): 1.0
Loss: 0.001270188600756228
Grad OK: cell.conv_h.weight | norm=0.000031
Gradient flow: True
Sanity check complete.


In [28]:
# Sanity check: Forward + backward on synthetic data

def generate_moving_dot(batch_size=2, seq_len=20, H=28, W=28, speed=1):
    """Generate simple synthetic moving dot sequence"""
    seq = torch.zeros(batch_size, seq_len, 1, H, W)
    for b in range(batch_size):
        x = np.random.randint(0, W)
        y = np.random.randint(0, H)
        vx = np.random.choice([-speed, 0, speed])
        vy = np.random.choice([-speed, 0, speed])
        for t in range(seq_len):
            seq[b, t, 0, y % H, x % W] = 1.0
            x += vx
            y += vy
    return seq

# Generate data
batch_size = 2
seq_len = 20
input_frames = 10
pred_len = seq_len - input_frames

seq = generate_moving_dot(batch_size=batch_size, seq_len=seq_len)
input_seq = seq[:, :input_frames]
target_seq = seq[:, input_frames:]

print(f"✓ Input shape: {input_seq.shape}, Target shape: {target_seq.shape}")

# Initialize model
from moving_mnist_models import Seq2SeqFERNN

model = Seq2SeqFERNN(
    input_channels=1,
    hidden_channels=16,
    height=28,
    width=28,
    v_range=2,
    decoder_conv_layers=1,
    pool_type='max',
    use_differentiable_flow=True
)
model.train()

# Forward pass
try:
    output, vel_probs = model(
        input_seq,
        pred_len=pred_len,
        teacher_forcing_ratio=0.0,
        target_seq=target_seq,
        return_vel_probs=True
    )
    print(f"✓ Forward pass successful")
    print(f"  Output shape: {output.shape}")
    print(f"  Vel probs shape: {vel_probs.shape}")
except Exception as e:
    print(f"✗ Forward pass failed: {e}")
    raise

# Loss & backward
try:
    criterion = torch.nn.MSELoss()
    loss = criterion(output, target_seq)
    print(f"✓ Loss computed: {loss.item():.6f}")
    
    loss.backward()
    print(f"✓ Backward pass successful")
except Exception as e:
    print(f"✗ Loss/backward failed: {e}")
    raise

# Check gradients
grad_ok = False
for name, p in model.named_parameters():
    if p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0:
        grad_ok = True
        print(f"✓ Gradient: {name} | norm={p.grad.norm().item():.6f}")
        break

print(f"\n{'='*50}")
print(f"✓✓ SANITY CHECK PASSED ✓✓" if grad_ok and loss.item() > 0 else "✗ Issues detected")
print(f"{'='*50}")

✓ Input shape: torch.Size([2, 10, 1, 28, 28]), Target shape: torch.Size([2, 10, 1, 28, 28])
✓ Forward pass successful
  Output shape: torch.Size([2, 10, 1, 28, 28])
  Vel probs shape: torch.Size([2, 20, 25])
✓ Loss computed: 0.001278
✓ Backward pass successful
✓ Gradient: velocity_predictor.feature_extractor.0.weight | norm=0.000000

✓✓ SANITY CHECK PASSED ✓✓


In [29]:
# Training stability check: 5 epochs on synthetic data

print("\n" + "="*60)
print("TRAINING STABILITY CHECK (5 epochs)")
print("="*60)

# Create fresh model and data
model = Seq2SeqFERNN(
    input_channels=1,
    hidden_channels=16,
    height=28,
    width=28,
    v_range=2,
    decoder_conv_layers=1,
    pool_type='max',
    use_differentiable_flow=True
)
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# Generate mini dataset (5 sequences with integer speeds)
torch.manual_seed(42)
train_seqs = [generate_moving_dot(batch_size=1, seq_len=20, speed=int(s)) for s in [0, 1, 1, 2, 0]]
input_seqs = torch.cat([s[:, :10] for s in train_seqs], dim=0)  # (5, 10, 1, 28, 28)
target_seqs = torch.cat([s[:, 10:] for s in train_seqs], dim=0)  # (5, 10, 1, 28, 28)

losses = []
for epoch in range(5):
    epoch_loss = 0
    
    # Mini-batch training
    for b in range(5):
        optimizer.zero_grad()
        
        output, _ = model(
            input_seqs[b:b+1],
            pred_len=10,
            teacher_forcing_ratio=0.5,
            target_seq=target_seqs[b:b+1],
            return_vel_probs=True
        )
        
        loss = criterion(output, target_seqs[b:b+1])
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / 5
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/5 | Loss: {avg_loss:.6f} | {'↓' if epoch > 0 and avg_loss < losses[epoch-1] else '↑'}")

# Check stability
print("\nStability Analysis:")
print(f"  Initial loss: {losses[0]:.6f}")
print(f"  Final loss:   {losses[-1]:.6f}")
print(f"  Trend:        {'STABLE ✓' if all(torch.isfinite(torch.tensor(l)) for l in losses) else 'NaN/Inf ✗'}")
print(f"  Decreasing:   {'Yes ✓' if losses[-1] < losses[0] else 'No (but may stabilize)'}")

if all(torch.isfinite(torch.tensor(l)) for l in losses) and losses[-1] < losses[0]:
    print("\n✓✓ TRAINING STABLE ✓✓")
elif all(torch.isfinite(torch.tensor(l)) for l in losses):
    print("\n✓ TRAINING STABLE (no divergence, loss may need more epochs to decrease)")
else:
    print("\n✗ TRAINING UNSTABLE (NaN/Inf detected)")
print(f"{model.use_differentiable_flow}")


TRAINING STABILITY CHECK (5 epochs)
Epoch 1/5 | Loss: 0.001203 | ↑
Epoch 2/5 | Loss: 0.001057 | ↓
Epoch 3/5 | Loss: 0.000887 | ↓
Epoch 4/5 | Loss: 0.000813 | ↓
Epoch 5/5 | Loss: 0.000621 | ↓

Stability Analysis:
  Initial loss: 0.001203
  Final loss:   0.000621
  Trend:        STABLE ✓
  Decreasing:   Yes ✓

✓✓ TRAINING STABLE ✓✓
True


In [ ]:
# Minimal end-to-end training + eval snippet for Seq2SeqFERNN
# Imports from the local package
import torch
from torch.utils.data import DataLoader
from moving_mnist_fp.moving_mnist_dataset import MovingMNISTDataset
from moving_mnist_fp.moving_mnist_models import Seq2SeqFERNN
from moving_mnist_fp.train_eval_utils import train_epoch, eval_epoch, eval_len_generalization

# Runtime config (small to fit in Colab / local GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
batch_size = 8
input_frames = 10
seq_len = 20

# Dataset + loader (small subset)
train_ds = MovingMNISTDataset(root='./data', train=True, seq_len=seq_len, image_size=28,
                              velocity_range_x=(-2,2), velocity_range_y=(-2,2), num_digits=1)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# Model (small hidden size to reduce memory)
model = Seq2SeqFERNN(input_channels=1,
                     hidden_channels=64,
                     height=28,
                     width=28,
                     h_kernel_size=3,
                     u_kernel_size=3,
                     v_range=2,
                     pool_type='max',
                     decoder_conv_layers=1).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# Run a short training loop (1 epoch) and evaluate
try:
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, input_frames, teacher_forcing_ratio=0.0, grad_clip=1.0)
    print('Train Loss:', train_loss)
    val_loss = eval_epoch(model, train_loader, criterion, device, input_frames, epoch=1, split_name='val')
    print('Val Loss:', val_loss)

    # Quick length-generalization check on a small subset
    gen_loader = DataLoader(MovingMNISTDataset(root='./data', train=False, seq_len=40, image_size=28,
                                               velocity_range_x=(-2,2), velocity_range_y=(-2,2), num_digits=1, random=False),
                            batch_size=batch_size, shuffle=False)
    mean, std = eval_len_generalization(model, gen_loader, device, input_frames)
    print('Len-gen mean (first 5):', mean[:5])
except RuntimeError as e:
    print('RuntimeError during train/eval:', e)
    if 'out of memory' in str(e).lower():
        print('OOM encountered. Try reducing batch_size, hidden_channels, or v_range and restart the cell.')

In [ ]:
# Full training + validation + test loop (notebook version)
import os
import torch
from torch.utils.data import DataLoader, random_split
from moving_mnist_fp.moving_mnist_dataset import MovingMNISTDataset
from moving_mnist_fp.moving_mnist_models import Seq2SeqFERNN
from moving_mnist_fp.train_eval_utils import train_epoch, eval_epoch, eval_len_generalization
from tqdm import tqdm

print("="*70)
print("FULL TRAINING/VALIDATION/TEST LOOP")
print("="*70)

# ============================================================================
# CONFIG
# ============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}\n')

# Hyperparameters
batch_size = 8
input_frames = 10
seq_len = 20
num_epochs = 10
lr = 1e-3
grad_clip = 1.0
teacher_forcing_ratio = 0.5
val_split = 0.1  # 10% for validation

# Model config
hidden_size = 64
v_range = 2
decoder_conv_layers = 1

print(f"Batch size: {batch_size}")
print(f"Epochs: {num_epochs}")
print(f"LR: {lr}")
print(f"Teacher forcing: {teacher_forcing_ratio}\n")

# ============================================================================
# LOAD DATA (train/val/test split)
# ============================================================================
print("Loading datasets...")
train_ds = MovingMNISTDataset(
    root='./data', 
    train=True, 
    seq_len=seq_len, 
    image_size=28,
    velocity_range_x=(-2, 2), 
    velocity_range_y=(-2, 2), 
    num_digits=1
)
test_ds = MovingMNISTDataset(
    root='./data', 
    train=False, 
    seq_len=seq_len, 
    image_size=28,
    velocity_range_x=(-2, 2), 
    velocity_range_y=(-2, 2), 
    num_digits=1,
    random=False
)

# Split train into train/val
val_size = int(val_split * len(train_ds))
train_size = len(train_ds) - val_size
train_split, val_split_ds = random_split(train_ds, [train_size, val_size])

train_loader = DataLoader(train_split, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_split_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print(f"Train: {len(train_split)}, Val: {len(val_split_ds)}, Test: {len(test_ds)}\n")

# ============================================================================
# MODEL & OPTIMIZER
# ============================================================================
print("Initializing model...")
model = Seq2SeqFERNN(
    input_channels=1,
    hidden_channels=hidden_size,
    height=28,
    width=28,
    h_kernel_size=3,
    u_kernel_size=3,
    v_range=v_range,
    pool_type='max',
    decoder_conv_layers=decoder_conv_layers
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = torch.nn.MSELoss()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}\n")

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_val_loss = float('inf')
train_losses = []
val_losses = []

print("Starting training...\n")
for epoch in range(1, num_epochs + 1):
    # Train
    try:
        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, device, 
            input_frames, teacher_forcing_ratio, grad_clip
        )
        train_losses.append(train_loss)
        print(f"Epoch {epoch}/{num_epochs} | Train Loss: {train_loss:.6f}", end=" | ")
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f"\n✗ OOM at epoch {epoch}. Reduce batch_size or hidden_size.")
            break
        raise
    
    # Validate
    try:
        val_loss = eval_epoch(
            model, val_loader, criterion, device, 
            input_frames, epoch=epoch, split_name='val'
        )
        val_losses.append(val_loss)
        print(f"Val Loss: {val_loss:.6f}", end="")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), './best_fernn_model.pth')
            print(" ✓ (best)")
        else:
            print()
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f"\n✗ OOM during validation.")
            break
        raise

print("\n" + "="*70)
print("TESTING")
print("="*70)

# Load best model
model.load_state_dict(torch.load('./best_fernn_model.pth'))
model.eval()

# Test
try:
    test_loss = eval_epoch(
        model, test_loader, criterion, device, 
        input_frames, epoch=0, split_name='test'
    )
    print(f"Test Loss: {test_loss:.6f}\n")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print(f"✗ OOM during testing.")
    raise

# ============================================================================
# LENGTH GENERALIZATION CHECK (optional, small subset)
# ============================================================================
print("="*70)
print("LENGTH GENERALIZATION CHECK")
print("="*70)

try:
    gen_ds = MovingMNISTDataset(
        root='./data', 
        train=False, 
        seq_len=40,  # Longer sequences
        image_size=28,
        velocity_range_x=(-2, 2), 
        velocity_range_y=(-2, 2), 
        num_digits=1,
        random=False
    )
    gen_loader = DataLoader(gen_ds, batch_size=batch_size, shuffle=False)
    
    mean, std = eval_len_generalization(model, gen_loader, device, input_frames)
    print(f"Mean MSE per timestep (first 10): {mean[:10]}")
    print(f"Std per timestep (first 10): {std[:10]}\n")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print("✗ OOM during length-gen eval. Skipping.")
    else:
        raise

# ============================================================================
# SUMMARY
# ============================================================================
print("="*70)
print("TRAINING SUMMARY")
print("="*70)
print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss:   {val_losses[-1]:.6f}")
print(f"Best val loss:    {best_val_loss:.6f}")
print(f"Test loss:        {test_loss:.6f}")
print(f"\nBest model saved: ./best_fernn_model.pth")